# Tools

In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("what is rag")
response.content

'<think>\nOkay, the user is asking "what is rag." First, I need to figure out what context they\'re referring to. "RAG" can stand for different things in various fields.\n\nLet me start by considering the most common meanings. In the context of AI and machine learning, RAG typically stands for Retrieval-Augmented Generation. That\'s a term I\'ve heard in discussions about large language models. It\'s a method where the model uses a retrieval system to gather information before generating a response, which helps in providing more accurate and up-to-date answers. That seems plausible, especially since the user might be referring to AI-related topics given the current trends.\n\nAnother possibility is in medical contexts, RAG could relate to something like "Recombinant Activated Factor VII" used in hemophilia treatment. But that\'s more niche and might not be what the user is asking about unless specified.\n\nIn computing, there\'s also the RAG (Red, Amber, Green) status system used in pr

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a given location"""
    return f"its cold in {location}"

model_with_tools=model.bind_tools([get_weather])

In [6]:
resoponse = model_with_tools.invoke("what is weather in germany?")
print(resoponse)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Germany. Let me check the tools available. There\'s a function called get_weather that requires a location parameter. Germany is a country, so I should use that as the location. I need to make sure to format the tool call correctly. The parameters should be a JSON object with "location" set to "Germany". Let me structure the response properly.\n', 'tool_calls': [{'id': '0qvznd4ey', 'function': {'arguments': '{"location":"Germany"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 153, 'total_tokens': 258, 'completion_time': 0.165354861, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.006064862, 'prompt_tokens_details': None, 'queue_time': 0.050045828, 'total_time': 0.171419723}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason':

# Tool Execution Loops

In [7]:
# step 1: Model generates tools calls
messages = [{"role":"user", "content":"what is the weather in germany"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Germany is currently cold. You might want to dress warmly if you're visiting or living there! ❄️


In [8]:
messages

[{'role': 'user', 'content': 'what is the weather in germany'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Germany. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Germany is the location here. I need to call that function with "Germany" as the argument. Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'y5qy0kgbm', 'function': {'arguments': '{"location":"Germany"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 153, 'total_tokens': 242, 'completion_time': 0.141199594, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.007958185, 'prompt_tokens_details': None, 'queue_time': 0.160512574, 'total_time': 0.149157779}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_r